In [1]:
#setup

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Proyek_CRM_KELOMPOK')
sys.path.append(str(PROJECT_DIR))

from project_config import RAW_DATA_PATH, CLEAN_DATA_PATH, OUTPUT_DIR

import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
#load dataset

df = pd.read_csv(RAW_DATA_PATH)

print('Shape awal:', df.shape)
df.head()

Shape awal: (388, 55)


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,...,Less Delivery time,High Quality of package,Number of calls,Politeness,Freshness,Temperature,Good Taste,Good Quantity,Output,Reviews
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,...,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Yes,Nil\n
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,...,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Very Important,Yes,Nil
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,...,Important,Very Important,Moderately Important,Very Important,Very Important,Important,Very Important,Moderately Important,Yes,"Many a times payment gateways are an issue, so..."
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,...,Very Important,Important,Moderately Important,Very Important,Very Important,Very Important,Very Important,Important,Yes,nil
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,...,Important,Important,Moderately Important,Important,Important,Important,Very Important,Very Important,Yes,NIL


In [3]:
#simpan kondisi awal

initial_rows = df.shape[0]
initial_columns = df.shape[1]
initial_duplicates = df.duplicated().sum()

In [4]:
# merapikan kolom

df.columns = df.columns.str.strip()

In [5]:
# merapikan nilai teks
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:
# ubah null menjadi string value
df = df.replace(['', 'nan', 'None', 'NULL', 'null'], np.nan)

In [7]:
#cek missing value
missing_report = (
    df.isnull().sum()
    .reset_index()
)

missing_report.columns = ['column_name', 'missing_count']
missing_report['missing_percent'] = missing_report['missing_count'] / len(df) * 100
missing_report = missing_report.sort_values('missing_count', ascending=False)

missing_report.head(20)

,column_name,missing_count,missing_percent
54,Reviews,1,0.257732
1,Gender,0,0.000000
0,Age,0,0.000000
3,Occupation,0,0.000000
4,Monthly Income,0,0.000000
5,Educational Qualifications,0,0.000000
2,Marital Status,0,0.000000
7,latitude,0,0.000000
8,longitude,0,0.000000
9,Pin code,0,0.000000


In [8]:
#simpan missing value
missing_path = OUTPUT_DIR / 'missing_value_report.csv'
missing_report.to_csv(missing_path, index=False)

print('Output disimpan:', missing_path)

Output disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/missing_value_report.csv


In [9]:
#hapus duplikat
df = df.drop_duplicates()

print('Shape setelah drop duplicate:', df.shape)

Shape setelah drop duplicate: (286, 55)


In [10]:
# buat target churn
df['churn_risk'] = df['Output'].map({
    'Yes': 0,
    'No': 1
})

df[['Output', 'churn_risk']].head()

,Output,churn_risk
0,Yes,0
1,Yes,0
2,Yes,0
3,Yes,0
4,Yes,0


In [11]:
#validasi target
df['churn_risk'].value_counts(dropna=False)

,count
churn_risk,
0,221
1,65


In [12]:
# buat laporan cleasing
cleaning_report = pd.DataFrame({
    'metric': [
        'rows_before_cleaning',
        'columns_before_cleaning',
        'duplicates_before_cleaning',
        'rows_after_cleaning',
        'columns_after_cleaning',
        'duplicates_after_cleaning',
        'target_missing_after_mapping'
    ],
    'value': [
        initial_rows,
        initial_columns,
        initial_duplicates,
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df['churn_risk'].isnull().sum()
    ]
})

cleaning_report

,metric,value
0,rows_before_cleaning,388
1,columns_before_cleaning,55
2,duplicates_before_cleaning,102
3,rows_after_cleaning,286
4,columns_after_cleaning,56
5,duplicates_after_cleaning,0
6,target_missing_after_mapping,0


In [13]:
# Simpan data clean dan report
df.to_csv(CLEAN_DATA_PATH, index=False)

cleaning_report_path = OUTPUT_DIR / 'cleaning_report.csv'
cleaning_report.to_csv(cleaning_report_path, index=False)

print('Data clean disimpan:', CLEAN_DATA_PATH)
print('Cleaning report disimpan:', cleaning_report_path)

Data clean disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/data/processed/online_delivery_clean.csv
Cleaning report disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/cleaning_report.csv
